# Used Car Resale Price Prediction - Model Training & Evaluation
In this notebook, we train, benchmark, and evaluate three regression algorithms:
1. **Random Forest Regressor**
2. **Extra Trees Regressor**
3. **Gradient Boosting Regressor**

Evaluation Metrics: **MAE**, **RMSE**, and **R² Score**.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.preprocess import prepare_datasets

sns.set_theme(style='whitegrid')

### 1. Load Preprocessed Training & Testing Data

In [2]:
X_train, X_test, y_train, y_test, preprocessor, feature_info = prepare_datasets('../data/Car details v3.csv')
print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

### 2. Define Algorithms & Train with Pipelines

In [3]:
models = {
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees Regressor': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'MAE (INR)': mae, 'RMSE (INR)': rmse, 'R2 Score': r2})
    predictions[name] = y_pred

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df

### 3. Visual Comparison: Actual vs Predicted Prices

In [4]:
best_preds = predictions['Random Forest Regressor'] / 1e5
y_test_lakh = y_test / 1e5

plt.figure(figsize=(8, 6))
plt.scatter(y_test_lakh, best_preds, alpha=0.5, color='teal')
plt.plot([0, 45], [0, 45], 'r--', linewidth=2, label='Ideal 45° Line')
plt.xlabel('Actual Price (Lakh Rs)')
plt.ylabel('Predicted Price (Lakh Rs)')
plt.title('Random Forest: Actual vs Predicted Price', fontsize=13, fontweight='bold')
plt.xlim(0, 45)
plt.ylim(0, 45)
plt.legend()
plt.show()